### Merging BLS Employment Projections with JOLTS, Anthropic AI Exposure, and O*NET Skills

This notebook walks through the full pipeline for enriching the BLS Employment Projections dataset (832 occupations) with three external data sources:

1. **BLS JOLTS** — industry-level hires, openings, and quits rates (2024 annual averages)
2. **Anthropic Economic Index** — theoretical and observed AI exposure by SOC major group, plus per-occupation values for the 10 most-exposed occupations
3. **O*NET Skills** — importance ratings (1–5) on 16 standardized skills, joined on the 6-digit SOC code

The output is a single tidy file with one row per occupation.

### Input files

All files are expected in the data/ directory alongside the notebook:

| File | Source | Purpose |
|---|---|---|
| `Employment_Projections.csv` | BLS Employment Projections 2024–2034 | Base dataset (832 occupations) |
| `soc_to_jolts_lookup.csv` | Hand-built crosswalk | Maps each SOC major group to one JOLTS industry |
| `jolts_2024_rates.csv` - BLS JOLTS news release tables 16/18/22 | Hires, openings, and quits rates by industry |
| `aei_exposure_by_soc_major_group.csv` | Anthropic Economic Index Figure 2 | Theoretical and observed AI exposure by SOC major group |
| `aei_top10_occupations.csv` | Anthropic Economic Index Figure 3 | Per-occupation overrides for the 10 most-exposed occupations |
| `skills_by_soc_full.csv` | O*NET Skills (matches with 752 of 832 occupations)

### Join architecture

The SOC occupation code is the spine. It joins directly to AEI on the full 6-digit code (for the top-10 overrides) and to O*NET Skills on the same 6-digit code, and joins to JOLTS through its 2-digit major-group prefix via the hand-built lookup to NAICS industries.

```
BLS Projections (SOC)
    │
    ├── SOC code ────────────────→ AEI top-10 (direct match)
    │                          └─→ O*NET Skills (direct match)
    │
    └── SOC major group (1st 2 digits) ───→ AEI by major group
                                       └─→ SOC→NAICS lookup → JOLTS
```



In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

### Load the base dataset

The BLS Employment Projections file is keyed by 6-digit SOC code. It contains employment levels for 2024 (the base year) and projected employment for 2034, along with wages and education attributes.

In [3]:
projections = pd.read_csv('data/Employment Projections.csv')
print(f'Rows: {len(projections)}')
print(f'Columns: {projections.columns.tolist()}')
projections.head(3)

Rows: 832
Columns: ['Occupation Title', 'Occupation Code', 'Employment 2024', 'Employment 2034', 'Employment Change, 2024-2034', 'Employment Percent Change, 2024-2034', 'Occupational Openings, 2024-2034 Annual Average', 'Median Annual Wage 2024', 'Typical Entry-Level Education', 'Education Code', 'Work Experience in a Related Occupation', 'Workex Code', 'Typical on-the-job Training', 'trCode']


,Occupation Title,Occupation Code,Employment 2024,Employment 2034,"Employment Change, 2024-2034","Employment Percent Change, 2024-2034","Occupational Openings, 2024-2034 Annual Average",Median Annual Wage 2024,Typical Entry-Level Education,Education Code,Work Experience in a Related Occupation,Workex Code,Typical on-the-job Training,trCode
0,Accountants and auditors * Account Auditor*...,13-2011,"1,579.80","1,652.60",72.8,4.6,124.2,"81,680",Bachelor's degree,3,NaN,4,NaN,6
1,Actors * Actor Understudy* Actress* Dramati...,27-2011,57,57.1,0.2,0.3,6.3,NaN,"Some college, no degree",6,NaN,4,Long-term on-the-job training,3
2,Actuaries * Actuarial Associate* Actuarial ...,15-2011,33.6,40.9,7.3,21.8,2.4,"125,770",Bachelor's degree,3,NaN,4,Long-term on-the-job training,3


### Derive the SOC major group

The major group is the first two characters of the SOC code. This becomes the join key for JOLTS and for the AEI major-group-level values.

In [4]:
projections['SOC_Major_Group'] = projections['Occupation Code'].str[:2]

# Sanity check: should be 22 standard SOC major groups
major_group_counts = projections['SOC_Major_Group'].value_counts().sort_index()
print(f'Distinct major groups: {len(major_group_counts)}')
major_group_counts

Distinct major groups: 22


SOC_Major_Group
11     38
13     32
15     21
17     36
19     48
21     17
23      8
25     64
27     41
29     71
31     17
33     24
35     17
37     10
39     32
41     22
43     54
45     14
47     60
49     51
51    105
53     50
Name: count, dtype: int64

### Build the JOLTS join

JOLTS publishes flow rates at the NAICS industry level, not by SOC. There is no published SOC-to-NAICS crosswalk that resolves to a single industry per occupation, so a manual mapping was built at the major-group level with three confidence tiers.

**Confidence semantics:**
- high — the major group's occupations concentrate in one NAICS sector (e.g., construction trades, healthcare practitioners)
- medium — concentrates in 2–3 sectors; the mapped one is the largest
- low — fully cross-industry; mapped to Total private as a least-bad fallback

In [6]:
soc_to_jolts = pd.read_csv('data/soc_to_jolts_lookup.csv', dtype={'SOC_Major_Group': str})
print(f'Mapping rows: {len(soc_to_jolts)}')
soc_to_jolts.head()

Mapping rows: 22


,SOC_Major_Group,SOC Major Group Name,JOLTS Industry (mapped),Mapping Confidence,Mapping Notes
0,11,Management,Total private,low,Management — spans all industries
1,13,Business and Financial Operations,Professional and business services,low,Business/Financial Ops — cross-industry
2,15,Computer and Mathematical,Information,medium,Computer & math — heavily Info & Prof services
3,17,Architecture and Engineering,Professional and business services,medium,Architecture/Engineering — prof services & mfg
4,19,"Life, Physical, and Social Science",Professional and business services,medium,Life/Physical/Social Science — research & cons...


### Load JOLTS 2024 rates

Source: BLS JOLTS news release tables 16 (job openings rate), 18 (hires rate), and 22 (quits rate), annual averages, not seasonally adjusted. March 2026 vintage.

In [8]:
jolts_2024 = pd.read_csv('data/jolts_2024_rates.csv')
print(f'JOLTS industries: {len(jolts_2024)}')
jolts_2024

JOLTS industries: 17


,JOLTS Industry (mapped),JOLTS Hires Rate 2024 (%),JOLTS Job Openings Rate 2024 (%),JOLTS Quits Rate 2024 (%)
0,Total private,3.7,4.8,2.3
1,Mining and logging,3.0,3.6,1.8
2,Construction,4.2,3.5,1.7
3,Manufacturing,2.6,3.6,1.6
4,"Trade, transportation, and utilities",3.6,3.4,2.3
5,Retail trade,4.0,3.2,2.7
6,"Transportation, warehousing, and utilities",4.1,4.3,2.2
7,Information,2.6,4.0,1.4
8,Financial activities,2.2,4.6,1.3
9,Professional and business services,4.4,5.5,2.4


### Execute the JOLTS join

Two-step merge: SOC major group → industry name → flow rates.

In [9]:
df = projections.merge(soc_to_jolts, on='SOC_Major_Group', how='left')
df = df.merge(jolts_2024, on='JOLTS Industry (mapped)', how='left')

# Validation
assert len(df) == len(projections), 'Row count changed — join inflated or dropped rows'
assert df['JOLTS Hires Rate 2024 (%)'].notna().all(), 'Some occupations failed to match a JOLTS industry'

print(f'Rows after JOLTS join: {len(df)}')
print('\nMapping confidence distribution:')
print(df['Mapping Confidence'].value_counts())

Rows after JOLTS join: 832

Mapping confidence distribution:
Mapping Confidence
high      409
medium    238
low       185
Name: count, dtype: int64


### Build the AI exposure join

Anthropic's Economic Index publishes:
- **Theoretical exposure (Eloundou β)** — share of tasks an LLM could plausibly perform
- **Observed exposure** — share of tasks where Claude usage is empirically observed

Both metrics are published at the SOC major-group level (Figure 2 of the underlying paper), with per-occupation overrides for the 10 most-exposed occupations (Figure 3). Per-occupation values for the full 756-SOC set live in `job_exposure.csv` on the Hugging Face release — see suggested next steps.

In [11]:
soc_ai_exposure = pd.read_csv('data/aei_exposure_by_soc_major_group.csv',
                              dtype={'SOC_Major_Group': str})
print(f'Major groups: {len(soc_ai_exposure)}')
soc_ai_exposure.head()

Major groups: 22


,SOC_Major_Group,SOC Major Group (AI exposure),AI Theoretical Exposure (%),AI Observed Exposure — Major Group (%),AI Displacement Risk Tier
0,11,Management,91.3,NaN,High
1,13,Business & Financial Operations,94.3,28.4,Very high
2,15,Computer & Mathematical,94.3,35.8,Very high
3,17,Architecture & Engineering,84.8,NaN,High
4,19,"Life, Physical & Social Sciences",77.0,NaN,Moderate


### Apply major-group-level AI exposure

In [12]:
df = df.merge(soc_ai_exposure, on='SOC_Major_Group', how='left')
assert len(df) == len(projections), 'Row count changed during AI exposure join'
print(f'Rows: {len(df)}, Columns: {df.shape[1]}')

Rows: 832, Columns: 26


### Apply per-occupation overrides for the top-10 most-exposed occupations

Anthropic's Figure 3 enumerates 10 individual occupations. Three of them (Computer Programmers, Customer Service Representatives, Data Entry Keyers) have explicitly cited exposure values; the other seven are flagged as in the top-10 but without specific percentages in the public report. The observed exposure is overridden where a number exists, and all 10 are tagged with a distinct risk tier.

In [13]:
top10 = pd.read_csv('data/aei_top10_occupations.csv')
top10_codes = set(top10['Occupation Code'])
top10_observed = dict(zip(top10['Occupation Code'],
                          top10['AI Observed Exposure — Occupation (%)']))

def resolve_observed_exposure(row):
    code = row['Occupation Code']
    override = top10_observed.get(code)
    if override is not None and not pd.isna(override):
        return override
    return row['AI Observed Exposure — Major Group (%)']

def resolve_risk_tier(row):
    if row['Occupation Code'] in top10_codes:
        return 'Very high (top-10 most exposed)'
    return row['AI Displacement Risk Tier']

df['AI Observed Exposure — Occupation (%)'] = df.apply(resolve_observed_exposure, axis=1)
df['AI Displacement Risk Tier'] = df.apply(resolve_risk_tier, axis=1)

print('Risk tier distribution:')
print(df['AI Displacement Risk Tier'].value_counts(dropna=False))

Risk tier distribution:
AI Displacement Risk Tier
Very low                           339
Moderate                           200
High                               144
Very high                           98
Low                                 41
Very high (top-10 most exposed)     10
Name: count, dtype: int64


### Build the O*NET skills join

Each row of `skills_by_soc_full.csv` is one occupation, keyed by the 6-digit SOC code, with one column per skill. Skill values are O*NET importance ratings on a 1–5 scale (higher = more central to the job). Empty cells indicate that O*NET marked the skill as Not Relevant for that occupation.

The join is a left merge on `Occupation Code`. The skills file's own `Occupation Title` column is dropped before merging to avoid colliding with the BLS title that's already in the dataframe.


In [14]:
skills = pd.read_csv('data/skills_by_soc_full.csv')
print(f'Skills file: {len(skills)} occupations × {skills.shape[1]} columns')

# Drop the skills file's own title column to avoid collision with the BLS title
skills = skills.drop(columns=['Occupation Title'])

# Track skill column names for later use (validation, column ordering)
skill_cols = [c for c in skills.columns if c != 'Occupation Code']
print(f'Skill columns to join: {len(skill_cols)}')

# Merge on Occupation Code (left join — preserves all 832 BLS rows;
# rows without skills data will have NaN in the new columns)
df = df.merge(skills, on='Occupation Code', how='left')

covered = df[skill_cols[0]].notna().sum()
print(f'Coverage: {covered}/{len(df)} occupations have skill data '
      f'({covered/len(df)*100:.1f}%)')

Skills file: 774 occupations × 37 columns
Skill columns to join: 35
Coverage: 751/832 occupations have skill data (90.3%)


## 6. Validation

Four invariants worth checking before publishing the merged file:

1. Row count unchanged from the source
2. No silent join failures (every occupation got a JOLTS industry)
3. Logical consistency: observed exposure ≤ theoretical exposure for any occupation that has both
4. Skills coverage matches expectation (demo file: ~15; full O*NET: ~830)

In [15]:
# 1. Row count invariant
assert len(df) == 832, f'Expected 832 rows, got {len(df)}'

# 2. JOLTS join completeness
assert df['JOLTS Industry (mapped)'].notna().all(), 'JOLTS join missing rows'
assert df['JOLTS Hires Rate 2024 (%)'].notna().all(), 'JOLTS rates missing rows'

# 3. Logical consistency on AEI: observed should not exceed theoretical
both = df[df['AI Theoretical Exposure (%)'].notna() &
          df['AI Observed Exposure — Occupation (%)'].notna()]
violations = both[both['AI Observed Exposure — Occupation (%)'] > both['AI Theoretical Exposure (%)']]
print(f'Observed > theoretical (expected: 0): {len(violations)}')
if len(violations):
    print(violations[['Occupation Title', 'AI Theoretical Exposure (%)',
                      'AI Observed Exposure — Occupation (%)']])

# Null distribution audit
print('\nNull counts by AI exposure column:')
print(df[['AI Theoretical Exposure (%)',
         'AI Observed Exposure — Occupation (%)',
         'AI Displacement Risk Tier']].isna().sum())

# 4. Skills coverage
skill_data_cols = [c for c in df.columns if c in skill_cols]
if skill_data_cols:
    covered = df[skill_data_cols[0]].notna().sum()
    print(f'\nSkills coverage: {covered}/{len(df)} occupations '
          f'({covered/len(df)*100:.1f}%)')

Observed > theoretical (expected: 0): 0

Null counts by AI exposure column:
AI Theoretical Exposure (%)              339
AI Observed Exposure — Occupation (%)    590
AI Displacement Risk Tier                  0
dtype: int64

Skills coverage: 751/832 occupations (90.3%)


## 7. Finalize and export

Drop intermediate helper columns and the major-group-level exposure (now superseded by the occupation-level column), then reorder for readability.

In [16]:
df = df.drop(columns=['SOC_Major_Group',
                       'AI Observed Exposure — Major Group (%)',
                       'SOC Major Group Name'], errors='ignore')

base_column_order = [
    'Occupation Title',
    'Occupation Code',
    'Employment 2024',
    'Employment 2034',
    'Employment Change, 2024-2034',
    'Employment Percent Change, 2024-2034',
    'Occupational Openings, 2024-2034 Annual Average',
    'Median Annual Wage 2024',
    'Typical Entry-Level Education',
    'JOLTS Industry (mapped)',
    'Mapping Confidence',
    'JOLTS Hires Rate 2024 (%)',
    'JOLTS Job Openings Rate 2024 (%)',
    'JOLTS Quits Rate 2024 (%)',
    'SOC Major Group (AI exposure)',
    'AI Theoretical Exposure (%)',
    'AI Observed Exposure — Occupation (%)',
    'AI Displacement Risk Tier',
    'Mapping Notes',
]
# Skill columns go after the base columns, sorted alphabetically
skill_cols_in_df = sorted([c for c in df.columns if c in skill_cols])
column_order = ([c for c in base_column_order if c in df.columns]
               + skill_cols_in_df)
df = df[column_order]

df.to_csv('Employment_Projections_with_JOLTS_AI_and_Skills.csv', index=False)
print(f'Wrote {len(df)} rows × {df.shape[1]} columns')
df.head(5)

Wrote 832 rows × 54 columns


,Occupation Title,Occupation Code,Employment 2024,Employment 2034,"Employment Change, 2024-2034","Employment Percent Change, 2024-2034","Occupational Openings, 2024-2034 Annual Average",Median Annual Wage 2024,Typical Entry-Level Education,JOLTS Industry (mapped),Mapping Confidence,JOLTS Hires Rate 2024 (%),JOLTS Job Openings Rate 2024 (%),JOLTS Quits Rate 2024 (%),SOC Major Group (AI exposure),AI Theoretical Exposure (%),AI Observed Exposure — Occupation (%),AI Displacement Risk Tier,Mapping Notes,Active Learning,Active Listening,Complex Problem Solving,Coordination,Critical Thinking,Equipment Maintenance,Equipment Selection,Installation,Instructing,Judgment and Decision Making,Learning Strategies,Management of Financial Resources,Management of Material Resources,Management of Personnel Resources,Mathematics,Monitoring,Negotiation,Operation and Control,Operations Analysis,Operations Monitoring,Persuasion,Programming,Quality Control Analysis,Reading Comprehension,Repairing,Science,Service Orientation,Social Perceptiveness,Speaking,Systems Analysis,Systems Evaluation,Technology Design,Time Management,Troubleshooting,Writing
0,Accountants and auditors * Account Auditor*...,13-2011,"1,579.80","1,652.60",72.8,4.6,124.2,"81,680",Bachelor's degree,Professional and business services,low,4.4,5.5,2.4,Business & Financial Operations,94.3,28.4,Very high (top-10 most exposed),Business/Financial Ops — cross-industry,3.12,3.75,3.38,3.25,3.75,1.00,1.25,1.12,2.75,3.50,2.75,2.25,1.88,2.75,3.25,3.38,2.88,1.38,1.62,1.75,3.00,2.00,1.88,3.88,1.0,1.25,3.12,3.12,3.75,2.88,2.88,1.75,3.12,1.00,3.50
1,Actors * Actor Understudy* Actress* Dramati...,27-2011,57,57.1,0.2,0.3,6.3,NaN,"Some college, no degree","Arts, entertainment, and recreation",medium,6.3,5.3,2.8,"Arts, Design, Entertainment & Media",83.7,19.2,High,Arts/Design/Entertainment/Media — mixed with Info,2.62,3.75,2.88,2.88,3.00,1.00,1.00,1.00,2.75,2.88,2.75,1.00,1.00,2.38,1.00,3.00,2.50,1.00,1.75,1.25,2.50,1.00,1.00,3.88,1.0,1.50,2.12,3.75,3.88,2.00,2.00,1.25,3.00,1.00,2.88
2,Actuaries * Actuarial Associate* Actuarial ...,15-2011,33.6,40.9,7.3,21.8,2.4,"125,770",Bachelor's degree,Information,medium,2.6,4.0,1.4,Computer & Mathematical,94.3,35.8,Very high,Computer & math — heavily Info & Prof services,3.25,4.00,4.00,3.00,4.25,1.00,1.00,1.00,2.75,4.25,2.88,2.62,2.00,2.88,4.25,3.12,2.75,1.00,3.00,2.00,2.88,2.12,1.88,4.25,1.0,2.00,3.00,3.00,3.88,3.88,4.00,1.88,3.00,1.00,3.50
3,Acupuncturists * Acupuncture Physician* Lic...,29-1291,15.3,16.4,1.0,6.8,0.9,"78,140",Master's degree,Health care and social assistance,high,3.4,6.5,2.2,Healthcare Practitioners & Technical,59.9,NaN,Moderate,Healthcare Practitioners,3.12,3.88,3.25,3.12,3.75,1.00,1.62,1.00,2.75,3.50,2.62,1.75,1.88,2.00,2.00,3.25,2.00,1.38,1.75,1.62,2.62,1.25,1.88,3.25,1.0,2.25,3.75,3.75,3.62,2.88,2.88,1.38,3.00,1.38,3.12
4,Adhesive bonding machine operators and tenders...,51-9191,12.2,12.3,0.1,1.0,1.3,"45,210",High school diploma or equivalent,Manufacturing,high,2.6,3.6,1.6,Production,NaN,NaN,Very low,Production occupations,2.62,3.12,2.75,3.00,2.88,2.88,2.12,1.12,2.25,2.62,2.25,1.75,1.75,1.88,2.50,3.12,1.88,3.88,1.75,3.75,2.00,1.62,3.12,3.00,3.0,1.50,2.25,2.88,3.12,2.75,2.12,1.25,3.00,3.00,2.75


## 8. Sanity-check the augmentation vs displacement pattern

One way to confirm the merge worked correctly: occupations flagged as very high AI exposure should split cleanly into displacement candidates (high exposure + projected decline) and augmentation candidates (high exposure + projected growth).

In [17]:
high_exposure = df[df['AI Displacement Risk Tier'].isin(
    ['Very high', 'Very high (top-10 most exposed)']
)]

print('=== Top 10 displacement candidates (high exposure + sharpest decline) ===')
display_cols = ['Occupation Title', 'Employment Percent Change, 2024-2034',
                'AI Observed Exposure — Occupation (%)', 'JOLTS Hires Rate 2024 (%)']
print(high_exposure.nsmallest(10, 'Employment Percent Change, 2024-2034')[display_cols]
      .to_string(index=False))

print('\n=== Top 10 augmentation candidates (high exposure + sharpest growth) ===')
print(high_exposure.nlargest(10, 'Employment Percent Change, 2024-2034')[display_cols]
      .to_string(index=False))

=== Top 10 displacement candidates (high exposure + sharpest decline) ===
                                                                                                                                                                                                                                                                          Occupation Title  Employment Percent Change, 2024-2034  AI Observed Exposure — Occupation (%)  JOLTS Hires Rate 2024 (%)
                                                                                                                                                        Word processors and typists    * Clerk Typist* Dictaphone Typist* Statistical Typist* Transcription Typist* Typist* Word Processor                                 -36.1                                   34.3                        3.7
                                                                                            Telephone operators    * 411 Directory Assistance Operator* 

## Sources

- **BLS Employment Projections, 2024–2034 release.** https://www.bls.gov/emp/
- **BLS JOLTS news release tables 16, 18, 22** (March 2026, annual averages, not seasonally adjusted). https://www.bls.gov/news.release/jolts.toc.htm
- **Anthropic Economic Index — Labor market impacts of AI** (Massenkoff & McCrory, 2026-03-05). https://www.anthropic.com/research/labor-market-impacts
- **Per-occupation AEI exposure data** (756 occupations) — release_2026_03_24/job_exposure.csv at https://huggingface.co/datasets/Anthropic/EconomicIndex
- **O*NET database** (skills, abilities, knowledge, tasks). https://www.onetcenter.org/database.html — download Skills.xlsx for the full ~900-occupation × 35-skill matrix.
- **Eloundou, T., Manning, S., Mishkin, P., & Rock, D. (2023).** *GPTs are GPTs: An Early Look at the Labor Market Impact Potential of Large Language Models.* arXiv:2303.10130.
